In [1]:
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import pandas as pd

energy_mix = pd.ExcelFile("DOE-538281345095_Energy_Mix.xlsx")
electricity_rates = pd.ExcelFile("DOE-538281345095_Electricity_Rates.xlsx")
customer_sales = pd.ExcelFile("DOE-538281345095_Customer_Sales_and_Count_20_25.xlsx")

#check the tabs in excel
print(f"Energy mix: {energy_mix.sheet_names}")
print(f"Electricty rates: {electricity_rates.sheet_names}")
print(f"Customer Sales: {customer_sales.sheet_names}")

Energy mix: ['Residential Non-Lifeline', 'Industrial Rates', 'Commercial Rates', 'NLResidential Rates', 'Sales and Customer Count', 'System Peak Demand', 'Visayas Sub Grid - System Peak', 'Power Generation Mix']
Electricty rates: ['Residential Non-Lifeline', 'Industrial Rates', 'Commercial Rates', 'NLResidential Rates']
Customer Sales: ['Residential Non-Lifeline', 'Sales and Customer Count']


Process the Electric rates

In [2]:
residential_nonlifeline_rates = pd.read_excel("DOE-538281345095_Electricity_Rates.xlsx", sheet_name="Residential Non-Lifeline")
# for residential areas that do not qualify for low-income assistance
# evaluates to NaN
residential_nlrates = pd.read_excel("DOE-538281345095_Electricity_Rates.xlsx", sheet_name="NLResidential Rates") # lifeline(?)
commercial_rates = pd.read_excel("DOE-538281345095_Electricity_Rates.xlsx", sheet_name="Commercial Rates")


#print(residential_nonlifeline_rates.head(), residential_nonlifeline_rates.shape)
print(f"Residential: {residential_nlrates.head()},"
      f" Shape: {residential_nlrates.shape},"
      f" Info: {residential_nlrates.info()},"
      f"Describe: {residential_nlrates.describe(include="all")},"
      f"Number of null: {residential_nlrates.isna().sum()}") # Shape (147, 78), 0 NaN
print(f"Commercial: {commercial_rates.head()}, Shape: {commercial_rates.shape}") # Shape ( 152, 78 )


C:\Users\DevelopmentONLY\PyCharmMiscProject\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


<class 'pandas.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 78 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   147 non-null    str   
 1   Unnamed: 1   147 non-null    str   
 2   Unnamed: 2   147 non-null    str   
 3   Unnamed: 3   147 non-null    str   
 4   Unnamed: 4   147 non-null    str   
 5   Unnamed: 5   147 non-null    str   
 6   2020         68 non-null     object
 7   Unnamed: 7   68 non-null     object
 8   Unnamed: 8   70 non-null     object
 9   Unnamed: 9   70 non-null     object
 10  Unnamed: 10  71 non-null     object
 11  Unnamed: 11  72 non-null     object
 12  Unnamed: 12  72 non-null     object
 13  Unnamed: 13  71 non-null     object
 14  Unnamed: 14  72 non-null     object
 15  Unnamed: 15  72 non-null     object
 16  Unnamed: 16  71 non-null     object
 17  Unnamed: 17  72 non-null     object
 18  2021         79 non-null     object
 19  Unnamed: 19  79 non-null     object
 20 

In [3]:
#Clean Data
res = residential_nlrates.copy()
res.columns = res.iloc[0]
res = res.iloc[1:].reset_index(drop=True)

com = commercial_rates.copy()
com.columns = com.iloc[0]
com = com.iloc[1:].reset_index(drop=True)
print(com.columns.tolist())
print(res.columns.tolist())

['Distribution Utility', 'ShortHand', 'GRID', 'Region', 'Island', 'EC/PDU', 'Jan-20', 'Feb-20', 'Mar-20', 'Apr-20', 'May-20', 'Jun-20', 'Jul-20', 'Aug-20', 'Sep-20', 'Oct-20', 'Nov-20', 'Dec-20', 'Jan-21', 'Feb-21', 'Mar-21', 'Apr-21', 'May-21', 'Jun-21', 'Jul-21', 'Aug-21', 'Sep-21', 'Oct-21', 'Nov-21', 'Dec-21', 'Jan-22', 'Feb-22', 'Mar-22', 'Apr-22', 'May-22', 'Jun-22', 'Jul-22', 'Aug-22', 'Sep-22', 'Oct-22', 'Nov-22', 'Dec-22', 'Jan-23', 'Feb-23', 'Mar-23', 'Apr-23', 'May-23', 'Jun-23', 'Jul-23', 'Aug-23', 'Sep-23', 'Oct-23', 'Nov-23', 'Dec-23', 'Jan-24', 'Feb-24', 'Mar-24', 'Apr-24', 'May-24', 'Jun-24', 'Jul-24', 'Aug-24', 'Sep-24', 'Oct-24', 'Nov-24', 'Dec-24', 'Jan-25', 'Feb-25', 'Mar-25', 'Apr-25', 'May-25', 'Jun-25', 'Jul-25', 'Aug-25', 'Sep-25', 'Oct-25', 'Nov-25', 'Dec-25']
['Distribution Utility', 'ShortHand', 'GRID', 'Region', 'Island', 'EC/PDU', 'Jan-20', 'Feb-20', 'Mar-20', 'Apr-20', 'May-20', 'Jun-20', 'Jul-20', 'Aug-20', 'Sep-20', 'Oct-20', 'Nov-20', 'Dec-20', 'Jan-21'

In [4]:
id_cols = ["Distribution Utility", "ShortHand", "GRID", "Region", "Island", "EC/PDU"]
month_cols = [
    col for col in res.columns
    if str(col).endswith("-20") or str(col).endswith("-24")  or str(col).endswith("-21")  or str(col).endswith("-25")
]
print(month_cols)


['Jan-20', 'Feb-20', 'Mar-20', 'Apr-20', 'May-20', 'Jun-20', 'Jul-20', 'Aug-20', 'Sep-20', 'Oct-20', 'Nov-20', 'Dec-20', 'Jan-21', 'Feb-21', 'Mar-21', 'Apr-21', 'May-21', 'Jun-21', 'Jul-21', 'Aug-21', 'Sep-21', 'Oct-21', 'Nov-21', 'Dec-21', 'Jan-24', 'Feb-24', 'Mar-24', 'Apr-24', 'May-24', 'Jun-24', 'Jul-24', 'Aug-24', 'Sep-24', 'Oct-24', 'Nov-24', 'Dec-24', 'Jan-25', 'Feb-25', 'Mar-25', 'Apr-25', 'May-25', 'Jun-25', 'Jul-25', 'Aug-25', 'Sep-25', 'Oct-25', 'Nov-25', 'Dec-25']


In [5]:
processed_cols_res = res.melt(
    id_vars= id_cols,
    value_vars = month_cols,
    var_name="Date",
    value_name="Rate",
)

processed_cols_com = com.melt(
    id_vars= id_cols,
    value_vars = month_cols,
    var_name="Date",
    value_name="Rate",
)

processed_cols_res["Customer Class"] = "Residential"
processed_cols_com["Customer Class"] = "Commercial"

rates = pd.concat(
    [processed_cols_res, processed_cols_com],
    ignore_index=True
)

rates["Rate"] = pd.to_numeric(
    rates["Rate"],
    errors="coerce"
)

rates["Date"] = pd.to_datetime(
    rates["Date"],
    format="%b-%y",
    errors="coerce"
)

rates_luzon = rates[
    rates["Island"].astype(str).str.strip().str.lower() == "luzon"
].copy()
print(rates_luzon) #final electricity rates data processed



                                    Distribution Utility  ShortHand      GRID  \
0              ABRA ELECTRIC COOPERATIVE, INC. (ABRECO)      ABRECO   ON-GRID   
1                          ANGELES ELECTRIC CORP. (AEC)         AEC   ON-GRID   
3       ALBAY ELECTRIC COOPERATIVE, INC./ALBAY POWER ...      ALECO   ON-GRID   
7           AURORA ELECTRIC COOPERATIVE, INC. (AURELCO)     AURELCO   ON-GRID   
10       BATANES ELECTRIC COOPERATIVE, INC. (BATANELCO)   BATANELCO  OFF-GRID   
...                                                  ...        ...       ...   
14241                        TARLAC ELECTRIC INC. (TEI)         TEI   ON-GRID   
14242  TABLAS ISLAND ELECTRIC COOPERATIVE, INC. (TIEL...     TIELCO  OFF-GRID   
14243  TICAO ISLAND ELECTRIC COOPERATIVE, INC. (TISEL...    TISELCO  OFF-GRID   
14246  ZAMBALES I ELECTRIC COOPERATIVE, INC. (ZAMECO I)    ZAMECO I   ON-GRID   
14247  ZAMBALES II ELECTRIC COOPERATIVE, INC. (ZAMECO...  ZAMECO II   ON-GRID   

                      Regio

Process Customer sales shit

In [32]:
raw = pd.read_excel("DOE-538281345095_Customer_Sales_and_Count_20_25.xlsx",  sheet_name="Sales and Customer Count",
    header=None)

print(raw.shape)

years = [2020, 2021, 2024, 2025]

customer_classes = [
    "Residential",
    "Commercial",
    "Industrial",
    "Others"
]

records = []

for year in years:

    sales_title = f"Customer Sales as of {year}"
    sales_title_rows = raw.index[
        raw.iloc[:, 1].astype(str).str.strip() == sales_title
    ]

    customer_title = f"Number of Customer as of {year}"
    customer_title_rows = raw.index[
        raw.iloc[:, 1].astype(str).str.strip() == customer_title
    ]

    if len(sales_title_rows) == 0:
        print(f"Sales section not found for {year}")
        continue

    if len(customer_title_rows) == 0:
        print(f"Customer section not found for {year}")
        continue

    sales_title_row = sales_title_rows[0]
    customer_title_row = customer_title_rows[0]

    sales_start = sales_title_row + 2

    customer_start = customer_title_row + 2

    for i, customer_class in enumerate(customer_classes):

        sales_row = sales_start + i
        customer_row = customer_start + i

        for month_num in range(1, 13):

            sales_value = raw.iloc[sales_row, month_num + 1]
            customer_value = raw.iloc[customer_row, month_num + 1]

            records.append({
                "Year": year,
                "Month": month_num,
                "Customer Class": customer_class,
                "Sales": pd.to_numeric(
                    sales_value,
                    errors="coerce"
                ),
                "Customer Count": pd.to_numeric(
                    customer_value,
                    errors="coerce"
                )
            })

sales_count_clean = pd.DataFrame(records)

print(sales_count_clean.head(20))
print(sales_count_clean.shape)


(101, 15)
    Year  Month Customer Class      Sales  Customer Count
0   2020      1    Residential  2507513.0               0
1   2020      2    Residential  2373081.0               0
2   2020      3    Residential  2413511.0               0
3   2020      4    Residential  2721300.0               0
4   2020      5    Residential  3405712.0               0
5   2020      6    Residential  3484903.0               0
6   2020      7    Residential  3027694.0               0
7   2020      8    Residential  2953173.0               0
8   2020      9    Residential  3102593.0               0
9   2020     10    Residential  2868256.0               0
10  2020     11    Residential  2728934.0               0
11  2020     12    Residential  2704915.0               0
12  2020      1     Commercial  2035222.0               0
13  2020      2     Commercial  2072947.0               0
14  2020      3     Commercial  1892171.0               0
15  2020      4     Commercial  1580908.0               0
16  

In [36]:
sales_count_clean["Date"] = pd.to_datetime(
    sales_count_clean["Year"].astype(str)
    + "-"
    + sales_count_clean["Month"].astype(str)
    + "-01"
)
# Sort
sales_count_final = sales_count_clean.sort_values(
    ["Date", "Customer Class"]
).reset_index(drop=True)

print(sales_count_final.head())
print(sales_count_final.tail())
print(sales_count_final["Year"].value_counts().sort_index())



   Year  Month Customer Class      Sales  Customer Count       Date
0  2020      1     Commercial  2035222.0               0 2020-01-01
1  2020      1     Industrial  2165032.0               0 2020-01-01
2  2020      1         Others   223778.0               0 2020-01-01
3  2020      1    Residential  2507513.0               0 2020-01-01
4  2020      2     Commercial  2072947.0               0 2020-02-01
     Year  Month Customer Class       Sales  Customer Count       Date
187  2025     11    Residential  3348754.35        20706258 2025-11-01
188  2025     12     Commercial  1765351.33         1586850 2025-12-01
189  2025     12     Industrial   804902.27          187773 2025-12-01
190  2025     12         Others   478473.32          382524 2025-12-01
191  2025     12    Residential  3150552.68        20843777 2025-12-01
Year
2020    48
2021    48
2024    48
2025    48
Name: count, dtype: int64


In [41]:
philippine_sales = sales_count_final[
    sales_count_final["Customer Class"].isin(
        ["Residential", "Commercial"]
    )
].copy()
print(philippine_sales["Customer Class"].value_counts())
#final sales is philippine_sales
annual_sales = (
    philippine_sales
    .groupby(["Year", "Customer Class"])["Sales"]
    .sum()
    .reset_index()
)

print(annual_sales)

Customer Class
Commercial     48
Residential    48
Name: count, dtype: int64
   Year Customer Class        Sales
0  2020     Commercial  20727103.00
1  2020    Residential  34291585.00
2  2021     Commercial  21118783.00
3  2021    Residential  34980664.00
4  2024     Commercial  28032924.00
5  2024    Residential  41205378.00
6  2025     Commercial  21422900.14
7  2025    Residential  40022203.93


Process Energy Mix

In [54]:
energy_file = pd.read_excel("DOE-538281345095_Energy_Mix.xlsx", header=None, sheet_name="Power Generation Mix")

print(energy_file.shape)
print(energy_file.head(20))

(102, 28)
                                                  0         1         2   \
0                               Department of Energy                       
1                              2024 POWER STATISTICS       NaN       NaN   
2     Gross Power Generation per Grid, by Plant Type       NaN       NaN   
3                                             In MWh                       
4                                                                          
5                                              LUZON      2003      2004   
6                   Generation per Technology (Grid)                       
7                                               Coal  14351121  15548335   
8                                          Oil-based   3595860   4590814   
9                                     Combined Cycle    438755    738437   
10                                            Diesel   2317101   2688194   
11                                       Gas Turbine      1737       183   
12

In [58]:
for i in range(len(energy_file)):
    row_text = " ".join(
        map(str, energy_file.iloc[i].tolist())
    )

    if "LUZON" in row_text.upper():
        print(i, row_text)

5 LUZON 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 % Share - 2025 nan nan nan
60 Luzon 37534630 39853911 40626730 41241457 43619911 44199534 44974855 50264725 50017259 52312426 54819517 56766481 60112863 66497549 68512419 72728369 76176710 72418709 75242541 79820536 83290439 90269376 85945490.35 0.7055444714075623 nan   nan
66 Luzon                                     1302991 1430583 1535520 1630613   0.772     nan


In [63]:
years = {
    "2020": 18,
    "2021": 19,
    "2024": 22,
    "2025": 23
}

top_level_sources = [
    "Coal",
    "Oil-based",
    "Natural Gas",
    "Renewable Energy (RE)"
]

energy_mix_luzon = energy_file.iloc[7:15, [0, 18, 19, 22, 23]].copy()

energy_mix_luzon.columns = [
    "Energy Source",
    "2020",
    "2021",
    "2024",
    "2025"
]

energy_mix_luzon["Energy Source"] = (
    energy_mix_luzon["Energy Source"]
    .astype(str)
    .str.strip()
)

for year in ["2020", "2021", "2024", "2025"]:
    energy_mix_luzon[year] = pd.to_numeric(
        energy_mix_luzon[year],
        errors="coerce"
    )

energy_mix_luzon = energy_mix_luzon[
    energy_mix_luzon["Energy Source"].isin(top_level_sources)
].copy()

print(energy_mix_luzon.to_string(index=False))

        Energy Source     2020     2021       2024        2025
                 Coal 40575727 43133237 57069815.0 48841547.75
            Oil-based  1804176   996287   602894.0   181079.18
          Natural Gas 19496927 19059682 18046729.0 21269050.26
Renewable Energy (RE) 10541880 12053335 14549938.0 15653813.16


In [64]:
total_generation = {
    "2020": 72418709,
    "2021": 75242541,
    "2024": 90269376,
    "2025": 85945490.35
}

for year in ["2020", "2021", "2024", "2025"]:
    energy_mix_luzon[f"Share {year} (%)"] = (
        energy_mix_luzon[year]
        / total_generation[year]
        * 100
    )

print(
    energy_mix_luzon.round(3).to_string(index=False)
)

        Energy Source     2020     2021       2024        2025  Share 2020 (%)  Share 2021 (%)  Share 2024 (%)  Share 2025 (%)
                 Coal 40575727 43133237 57069815.0 48841547.75          56.029          57.326          63.222          56.829
            Oil-based  1804176   996287   602894.0   181079.18           2.491           1.324           0.668           0.211
          Natural Gas 19496927 19059682 18046729.0 21269050.26          26.923          25.331          19.992          24.747
Renewable Energy (RE) 10541880 12053335 14549938.0 15653813.16          14.557          16.019          16.118          18.214


In [65]:
energy_mix_final = energy_mix_luzon.melt(
    id_vars="Energy Source",
    value_vars=[
        "Share 2020 (%)",
        "Share 2021 (%)",
        "Share 2024 (%)",
        "Share 2025 (%)"
    ],
    var_name="Year",
    value_name="Share"
)

energy_mix_final["Year"] = (
    energy_mix_final["Year"]
    .str.extract(r"(\d{4})")
    .astype(int)
)

print(energy_mix_final.to_string(index=False))

        Energy Source  Year     Share
                 Coal  2020 56.029343
            Oil-based  2020  2.491312
          Natural Gas  2020 26.922500
Renewable Energy (RE)  2020 14.556846
                 Coal  2021 57.325599
            Oil-based  2021  1.324101
          Natural Gas  2021 25.330992
Renewable Energy (RE)  2021 16.019309
                 Coal  2024 63.221679
            Oil-based  2024  0.667883
          Natural Gas  2024 19.992083
Renewable Energy (RE)  2024 16.118354
                 Coal  2025 56.828517
            Oil-based  2025  0.210691
          Natural Gas  2025 24.747139
Renewable Energy (RE)  2025 18.213653


Comparison HERE